# Battery Cell Sweep: 100+ Material Pairings Scored

This notebook loads the results from `showcase/battery_cell_sweep.py` and visualizes:
- Cathode vs Electrolyte heatmap
- Cathode vs Anode heatmap
- Score distribution histograms
- Top 10 / Bottom 10 pairs

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import json

# Load results (run 'python -m showcase.battery_cell_sweep' first if needed)
csv_path = '../showcase/outputs/battery_pairwise_scores.csv'
json_path = '../showcase/outputs/battery_sweep.json'

if not os.path.exists(csv_path):
    print('Generating data...')
    from showcase.battery_cell_sweep import main
    main()

df = pd.read_csv(csv_path)
with open(json_path) as f:
    summary = json.load(f)

print(f"Total pairs: {len(df)}")
print(f"Viable: {df['viable'].sum()}")
print(f"Non-viable: {(~df['viable']).sum()}")
df.head(10)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Cathode-Electrolyte Heatmap
ce = df[df['pair_type'] == 'cathode-electrolyte'].copy()
pivot_ce = ce.pivot(index='material_a', columns='material_b', values='total_score')

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(pivot_ce.values, cmap='RdYlGn', vmin=0.3, vmax=1.0, aspect='auto')
ax.set_xticks(range(len(pivot_ce.columns)))
ax.set_xticklabels(pivot_ce.columns, rotation=45, ha='right')
ax.set_yticks(range(len(pivot_ce.index)))
ax.set_yticklabels(pivot_ce.index)
ax.set_title('Cathode-Electrolyte Compatibility Scores')
plt.colorbar(im, label='Score (0-1)')

# Add score text
for i in range(len(pivot_ce.index)):
    for j in range(len(pivot_ce.columns)):
        val = pivot_ce.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# Cathode-Anode Heatmap
ca = df[df['pair_type'] == 'cathode-anode'].copy()
pivot_ca = ca.pivot(index='material_a', columns='material_b', values='total_score')

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot_ca.values, cmap='RdYlGn', vmin=0.3, vmax=1.0, aspect='auto')
ax.set_xticks(range(len(pivot_ca.columns)))
ax.set_xticklabels(pivot_ca.columns, rotation=45, ha='right')
ax.set_yticks(range(len(pivot_ca.index)))
ax.set_yticklabels(pivot_ca.index)
ax.set_title('Cathode-Anode Compatibility Scores')
plt.colorbar(im, label='Score (0-1)')

for i in range(len(pivot_ca.index)):
    for j in range(len(pivot_ca.columns)):
        val = pivot_ca.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Score Distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, ptype in zip(axes, ['cathode-electrolyte', 'cathode-anode', 'anode-electrolyte']):
    subset = df[df['pair_type'] == ptype]['total_score'].dropna()
    ax.hist(subset, bins=15, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(0.5, color='red', linestyle='--', label='Viability threshold')
    ax.set_title(ptype)
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.suptitle('Score Distributions by Interface Type', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 and Bottom 10
print('TOP 10 PAIRINGS')
print('=' * 70)
for _, row in df.nlargest(10, 'total_score').iterrows():
    status = 'VIABLE' if row['viable'] else 'FAIL'
    print(f"  {row['material_a']:10s} + {row['material_b']:10s} "
          f"({row['pair_type']:20s}) = {row['total_score']:.4f} [{status}]")

print()
print('BOTTOM 10 PAIRINGS')
print('=' * 70)
scored = df.dropna(subset=['total_score'])
for _, row in scored.nsmallest(10, 'total_score').iterrows():
    status = 'VIABLE' if row['viable'] else 'FAIL'
    print(f"  {row['material_a']:10s} + {row['material_b']:10s} "
          f"({row['pair_type']:20s}) = {row['total_score']:.4f} [{status}]")